# Module 04 — ML Theory & Evaluation
## 04-08: Convex Optimization Foundations

**Objective:** Implement convexity checks, the Lagrangian dual, and KKT conditions from scratch, then derive and solve the soft-margin SVM dual — connecting abstract optimization theory to a concrete ML algorithm.

**Prerequisites:** 1-09 (Numerical Methods), 2-05 (Support Vector Machines)

## Part 0 — Setup & Prerequisites

Convex optimization is the mathematical engine beneath SVMs, logistic regression,
and constrained RL policy optimization. This notebook covers:

1. **Convex sets and functions** — formal definitions and verification algorithms.
2. **Lagrangian duality** — converting constrained problems to unconstrained dual problems.
3. **KKT conditions** — necessary and sufficient conditions at the optimal solution.
4. **SVM dual** — derive the dual from the primal Lagrangian and solve it via scipy.

> **Note:** `scipy` is permitted in Module 4. The SVM dual is solved with `scipy.optimize.minimize`
> (SLSQP), not a black-box QP solver, so the connection to the Lagrangian is explicit.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
import math
from typing import Any, Callable
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

from scipy.optimize import minimize as scipy_minimize
from scipy.optimize import linprog
from sklearn.datasets import make_classification, make_blobs
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
N_SVM        = 200     # classification samples for SVM dual experiment
C_SVM        = 1.0     # soft-margin SVM regularisation parameter
N_CONVEX_CHK = 200     # number of t values to sample for convexity checks
N_GRID_1D    = 400     # 1-D grid resolution for function visualization
KKT_TOL      = 1e-5    # KKT condition tolerance
DUAL_TOL     = 1e-4    # dual variable threshold for support vector detection


In [ ]:
# ── Data Generation ───────────────────────────────────────────────────────────
# 1. Binary classification for SVM dual (linearly separable with some noise)
X_svm_raw, y_svm_bin = make_classification(
    n_samples=N_SVM, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.5, flip_y=0.02, random_state=SEED,
)
scaler_svm  = StandardScaler()
X_svm       = scaler_svm.fit_transform(X_svm_raw)
y_svm       = 2 * y_svm_bin - 1  # convert {0,1} -> {-1,+1}
X_tr_svm, X_te_svm, y_tr_svm, y_te_svm = train_test_split(
    X_svm, y_svm, test_size=0.25, random_state=SEED,
)

# 2. 1-D domain for visualising convex functions
x_1d = np.linspace(-3.0, 3.0, N_GRID_1D)

# 3. Simple 2-D LP data points for feasible region visualization
lp_vertices = np.array([[0, 0], [2, 0], [2, 2], [0, 4]], dtype=float)

print(f"SVM data   : {X_tr_svm.shape[0]} train | {X_te_svm.shape[0]} test  "
      f"(2 features, y in {{-1,+1}})")
print(f"Pos class  : {int((y_tr_svm == +1).sum())}  |  Neg class: {int((y_tr_svm == -1).sum())}")
print(f"1-D grid   : {len(x_1d)} points in [{x_1d[0]:.1f}, {x_1d[-1]:.1f}]")


---
## Part 1 — Theoretical Foundation & Implementation

### 1.1 Convex Sets

A set $\mathcal{C}$ is **convex** if for any $x, y \in \mathcal{C}$ and $t \in [0,1]$:
$$tx + (1-t)y \in \mathcal{C}$$

**Examples of convex sets:** hyperplanes, halfspaces, balls (Euclidean, $\ell_1$, $\ell_\infty$),
polyhedra ($\{x \mid Ax \preceq b\}$), positive semidefinite cone.

**Preservation operations:** intersection of convex sets is convex; affine images are convex.

### 1.2 Convex Functions

A function $f: \mathbb{R}^n \to \mathbb{R}$ is **convex** if its domain is convex and:
$$f(tx + (1-t)y) \leq t\,f(x) + (1-t)\,f(y) \quad \forall x,y,\, t \in [0,1]$$

**Equivalent conditions (when $f$ is smooth):**
- **First-order:** $f(y) \geq f(x) + \nabla f(x)^\top (y - x)$ — tangent plane is global underestimator.
- **Second-order:** $\nabla^2 f(x) \succeq 0$ for all $x$ — Hessian is positive semidefinite.


### 1.3 Implementing Convexity Checks

We implement a statistical convexity checker for sets and functions, then verify
on known convex/concave examples before applying to the optimization problems.


In [ ]:
# ── Part 1: Convex Sets ────────────────────────────────────────────────────────


def is_convex_set_check(
    membership_fn: Callable[[np.ndarray], bool],
    p1: np.ndarray,
    p2: np.ndarray,
    n_checks: int = 200,
) -> tuple[bool, float]:
    '''Statistically check if convex combinations of two points lie in a set.

    Samples n_checks values t in [0, 1] and verifies that t*p1 + (1-t)*p2
    satisfies membership_fn. If all combinations are in the set, the set
    passes this pair-wise convexity test.

    Args:
        membership_fn: Callable(np.ndarray) -> bool, returns True if point is in set.
        p1: First point, shape (n_dims,).
        p2: Second point, shape (n_dims,).
        n_checks: Number of t values to test (more = more thorough).

    Returns:
        Tuple (all_in_set, fraction_in_set).
    '''
    t_vals   = np.linspace(0.0, 1.0, n_checks)
    count_in = 0
    for t in t_vals:
        midpoint = t * p1 + (1.0 - t) * p2
        if membership_fn(midpoint):
            count_in += 1
    frac = count_in / n_checks
    return count_in == n_checks, frac


# ── Define three example sets ─────────────────────────────────────────────────
# Convex set 1: unit ball {x : ||x||_2 <= 1}
ball_membership    = lambda p: float(np.linalg.norm(p)) <= 1.0 + 1e-9

# Convex set 2: halfspace {x : x[0] + x[1] <= 1}
halfspace_membership = lambda p: float(p[0] + p[1]) <= 1.0 + 1e-9

# Non-convex set: annulus {x : 0.5 <= ||x||_2 <= 1}
annulus_membership = lambda p: 0.5 - 1e-9 <= float(np.linalg.norm(p)) <= 1.0 + 1e-9

# ── Run convexity checks on random point pairs ────────────────────────────────
rng_cvx = np.random.default_rng(SEED)

sets_info = [
    ("Unit ball (convex)",    ball_membership,       True),
    ("Halfspace (convex)",    halfspace_membership,  True),
    ("Annulus (non-convex)",  annulus_membership,    False),
]

print("Convex set verification (200 t-samples per pair, 50 random pairs):")
print(f"{'Set':>25s}  {'Expected':>8s}  {'Frac_in':>8s}  {'Correct?':>9s}")
print("-" * 58)

for set_name, mem_fn, expected_convex in sets_info:
    fracs: list[float] = []
    all_passes = True
    for _ in range(50):
        p1_test = rng_cvx.uniform(-1, 1, size=2)
        p2_test = rng_cvx.uniform(-1, 1, size=2)
        # Only test pairs where both endpoints are in the set
        if mem_fn(p1_test) and mem_fn(p2_test):
            ok, frac = is_convex_set_check(mem_fn, p1_test, p2_test, n_checks=N_CONVEX_CHK)
            fracs.append(frac)
            if not ok:
                all_passes = False
    mean_frac = float(np.mean(fracs)) if fracs else float("nan")
    correct   = (all_passes == expected_convex) or (not expected_convex and not all_passes)
    print(f"  {set_name:>23s}  {'convex':>8s}  {mean_frac:8.4f}  {'YES' if correct else 'NO':>9s}")

print("")
print("Annulus: some convex combinations (chord between inner-ring points) leave the set.")


### 1.4 Convex Functions — Jensen's Inequality and Second-Order Condition

We implement two complementary checks:
- **Jensen check:** verify $f(tx + (1-t)y) \leq tf(x) + (1-t)f(y)$ for sampled pairs.
- **Second-order check:** verify $f''(x) \geq 0$ (for 1-D) or $\nabla^2 f \succeq 0$.


In [ ]:
# ── Part 1: Convex Function Checks ────────────────────────────────────────────


def check_jensen_inequality(
    f: Callable[[float], float],
    x1: float,
    x2: float,
    n_samples: int = 100,
    tol: float = 1e-8,
) -> tuple[bool, float]:
    '''Verify Jensen's inequality for a 1-D function on [x1, x2].

    Checks f(tx + (1-t)y) <= t*f(x) + (1-t)*f(y) for n_samples values of t in [0,1].
    The maximum violation is returned as a diagnostic.

    Args:
        f: 1-D function R -> R.
        x1: Left endpoint of the domain interval.
        x2: Right endpoint of the domain interval.
        n_samples: Number of t values to test.
        tol: Tolerance for the inequality (allow small numerical errors).

    Returns:
        Tuple (satisfies_jensen, max_violation) where max_violation >= 0 means convex.
    '''
    t_vals     = np.linspace(0.0, 1.0, n_samples)
    violations = []
    for t in t_vals:
        mid      = t * x1 + (1.0 - t) * x2
        lhs      = f(mid)
        rhs      = t * f(x1) + (1.0 - t) * f(x2)
        violations.append(lhs - rhs)
    max_viol = float(max(violations))
    return max_viol <= tol, max_viol


def numerical_second_derivative(
    f: Callable[[float], float],
    x: float,
    h: float = 1e-5,
) -> float:
    '''Compute the second derivative of f at x via central finite differences.

    Args:
        f: 1-D function R -> R.
        x: Point at which to evaluate the second derivative.
        h: Finite difference step size.

    Returns:
        Approximate f''(x) using (f(x+h) - 2f(x) + f(x-h)) / h^2.
    '''
    return (f(x + h) - 2.0 * f(x) + f(x - h)) / (h ** 2)


def check_second_order_convexity(
    f: Callable[[float], float],
    x_samples: np.ndarray,
    tol: float = -1e-6,
) -> tuple[bool, float]:
    '''Check the second-order convexity condition f''(x) >= 0 at sampled points.

    Args:
        f: 1-D function R -> R (twice differentiable).
        x_samples: Array of points at which to evaluate f''.
        tol: Minimum allowed second derivative (allows small numerical noise).

    Returns:
        Tuple (all_nonneg, min_second_deriv).
    '''
    second_derivs = [numerical_second_derivative(f, float(x)) for x in x_samples]
    min_d2 = float(min(second_derivs))
    return min_d2 >= tol, min_d2


# ── Common convex / concave functions ────────────────────────────────────────
convex_fns: list[dict] = [
    {"name": "x^2 (convex)",       "f": lambda x: float(x) ** 2,       "convex": True},
    {"name": "exp(x) (convex)",    "f": lambda x: math.exp(float(x)),   "convex": True},
    {"name": "|x| (convex)",       "f": lambda x: abs(float(x)),        "convex": True},
    {"name": "max(0,x) (convex)",  "f": lambda x: max(0.0, float(x)),   "convex": True},
    {"name": "-x^2 (concave)",     "f": lambda x: -(float(x) ** 2),     "convex": False},
    {"name": "sin(x) (neither)",   "f": lambda x: math.sin(float(x)),   "convex": False},
]

print("Convex function verification — Jensen's inequality and f''(x) >= 0:")
hdr_fn = "Function"; hdr_ex = "Expected"; hdr_jn = "Jensen OK"; hdr_d2 = "f''>=0"; hdr_ok = "Correct?"
print(f"{hdr_fn:>22s}  {hdr_ex:>8s}  {hdr_jn:>10s}  {hdr_d2:>10s}  {hdr_ok:>9s}")
print("-" * 68)

x_chk_samples = np.linspace(-2.0, 2.0, 50)
for fn_info in convex_fns:
    f_fn   = fn_info["f"]
    ok_j, max_viol = check_jensen_inequality(f_fn, -2.0, 2.0, n_samples=N_CONVEX_CHK)
    ok_d2, min_d2  = check_second_order_convexity(f_fn, x_chk_samples)
    overall_ok = (ok_j == fn_info["convex"])
    print(f"  {fn_info['name']:>20s}  {'convex' if fn_info['convex'] else 'non-cvx':>8s}"
          f"  {'YES' if ok_j else 'NO ':>10s}  {'YES' if ok_d2 else 'NO ':>10s}"
          f"  {'YES' if overall_ok else 'NO ':>9s}")

print("")
print("Jensen's inequality is the defining property. Second-order is sufficient (smooth fns).")


In [ ]:
# ── Visualise Convex Functions ─────────────────────────────────────────────────
fig_cvx, axes_cvx = plt.subplots(1, 2, figsize=(14, 5))

# Left: common convex functions on [-2, 2]
x_plot_cvx = np.linspace(-2.5, 2.5, 300)
plot_fns = [
    ("x^2",       lambda x: x ** 2,           "steelblue"),
    ("exp(x)",    lambda x: np.exp(x),         "green"),
    ("|x|",       lambda x: np.abs(x),         "orange"),
    ("max(0,x)",  lambda x: np.maximum(0, x),  "purple"),
]
for label_c, fn_c, col_c in plot_fns:
    axes_cvx[0].plot(x_plot_cvx, fn_c(x_plot_cvx), label=label_c, color=col_c, lw=2)

axes_cvx[0].set_xlabel("x")
axes_cvx[0].set_ylabel("f(x)")
axes_cvx[0].set_title("Common Convex Functions", fontsize=11, fontweight="bold")
axes_cvx[0].legend(fontsize=9)
axes_cvx[0].set_ylim(-0.5, 6)
axes_cvx[0].axhline(0, color="gray", lw=0.8)
axes_cvx[0].axvline(0, color="gray", lw=0.8)

# Right: Jensen's inequality visualised for x^2
x_a, x_b = -1.5, 1.5
f_demo    = lambda x: x ** 2
t_demo    = np.linspace(0, 1, 80)
mix_pts   = t_demo * x_a + (1 - t_demo) * x_b
lhs_vals  = f_demo(mix_pts)
rhs_vals  = t_demo * f_demo(x_a) + (1 - t_demo) * f_demo(x_b)

axes_cvx[1].plot(x_plot_cvx, f_demo(x_plot_cvx), color="steelblue", lw=2.5, label="f(x) = x^2")
axes_cvx[1].plot(mix_pts, rhs_vals, color="tomato", lw=2, ls="--",
                  label="Chord t*f(a)+(1-t)*f(b)")
axes_cvx[1].plot(mix_pts, lhs_vals, color="green", lw=2, ls=":",
                  label="f(tx+(1-t)y) (on curve)")
axes_cvx[1].fill_between(mix_pts, lhs_vals, rhs_vals, alpha=0.15, color="tomato")
axes_cvx[1].scatter([x_a, x_b], [f_demo(x_a), f_demo(x_b)], c="black", s=60, zorder=5)
axes_cvx[1].set_xlabel("x")
axes_cvx[1].set_ylabel("f(x)")
axes_cvx[1].set_title("Jensen's Inequality: chord >= curve for convex f",
                       fontsize=11, fontweight="bold")
axes_cvx[1].legend(fontsize=9)

plt.suptitle("Convex Functions: Definition and Examples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Chord (red dashed) lies ABOVE the function curve (green dotted) — Jensen satisfied.")


---
### 1.5 Lagrangian Duality

For the primal problem:
$$
\min_{x} f(x) \quad \text{s.t.} \quad g_i(x) \leq 0,\; h_j(x) = 0
$$

The **Lagrangian** is:
$$
L(x, \boldsymbol{\lambda}, \boldsymbol{\nu}) = f(x) + \sum_i \lambda_i g_i(x) + \sum_j \nu_j h_j(x)
$$

with $\lambda_i \geq 0$ (**dual feasibility**).

The **dual function** $g(\boldsymbol{\lambda}, \boldsymbol{\nu}) = \inf_x L(x, \boldsymbol{\lambda}, \boldsymbol{\nu})$
is always concave (infimum of affine functions). It provides a **lower bound** on the primal optimum:
$$
g(\boldsymbol{\lambda}, \boldsymbol{\nu}) \leq p^* \quad \forall\, \lambda \succeq 0
$$

The **dual problem** maximises this lower bound:
$$
\max_{\boldsymbol{\lambda} \succeq 0, \boldsymbol{\nu}} g(\boldsymbol{\lambda}, \boldsymbol{\nu})
$$

**Strong duality** ($d^* = p^*$) holds for convex problems when Slater's condition is satisfied.


In [ ]:
# ── Part 1: Lagrangian Duality ─────────────────────────────────────────────────


def lagrangian_value(
    f_val: float,
    g_vals: np.ndarray,
    h_vals: np.ndarray,
    lam: np.ndarray,
    nu: np.ndarray,
) -> float:
    '''Compute the Lagrangian L(x, lambda, nu) from pre-evaluated function values.

    L = f(x) + lambda^T g(x) + nu^T h(x)

    Args:
        f_val: Scalar objective value f(x).
        g_vals: Inequality constraint values g_i(x), shape (n_ineq,).
        h_vals: Equality constraint values h_j(x), shape (n_eq,).
        lam: Inequality dual variables lambda >= 0, shape (n_ineq,).
        nu: Equality dual variables nu (unconstrained), shape (n_eq,).

    Returns:
        Scalar Lagrangian value.
    '''
    lag = float(f_val)
    if lam.size > 0:
        lag += float(lam @ g_vals)
    if nu.size > 0:
        lag += float(nu @ h_vals)
    return lag


def dual_function_approx(
    lam: np.ndarray,
    nu: np.ndarray,
    f: Callable[[np.ndarray], float],
    g_fns: list[Callable[[np.ndarray], float]],
    h_fns: list[Callable[[np.ndarray], float]],
    x_grid: np.ndarray,
) -> float:
    '''Approximate the dual function g(lambda, nu) by minimising L over a discrete x grid.

    g(lambda, nu) = inf_x L(x, lambda, nu)

    This approximation replaces the true infimum over all x with the minimum over
    a finite grid of candidate x values. Useful for low-dimensional problems only.

    Args:
        lam: Inequality dual variables, shape (n_ineq,).
        nu: Equality dual variables, shape (n_eq,).
        f: Objective function Callable, x -> scalar.
        g_fns: List of inequality constraint Callables, each x -> scalar.
        h_fns: List of equality constraint Callables, each x -> scalar.
        x_grid: Candidate x values, shape (n_grid, n_vars) or (n_grid,) for 1-D.

    Returns:
        Approximate dual value g(lambda, nu).
    '''
    min_lag = float("inf")
    for x_val in x_grid:
        f_val  = float(f(x_val))
        g_vals = np.array([float(g(x_val)) for g in g_fns]) if g_fns else np.array([])
        h_vals = np.array([float(h(x_val)) for h in h_fns]) if h_fns else np.array([])
        lag    = lagrangian_value(f_val, g_vals, h_vals, lam, nu)
        if lag < min_lag:
            min_lag = lag
    return min_lag


# ── Example 1: Equality-constrained QP ────────────────────────────────────────
# min x1^2 + x2^2  s.t.  x1 + x2 = 1
# Lagrangian: L = x1^2 + x2^2 + nu*(x1+x2-1)
# Stationarity: 2*x1 + nu = 0, 2*x2 + nu = 0  -> x1=x2=-nu/2
# Substituting: g(nu) = -nu^2/2 - nu  (dual function)
# Maximize g: g'(nu) = -nu - 1 = 0 -> nu* = -1
# g(-1) = -1/2 + 1 = 1/2
# Primal: x1* = x2* = 1/2, f* = 1/4 + 1/4 = 1/2  -> p* = d* = 1/2

result_qp_eq = scipy_minimize(
    lambda x: float(x[0] ** 2 + x[1] ** 2),
    x0=np.array([0.5, 0.5]),
    jac=lambda x: np.array([2.0 * x[0], 2.0 * x[1]]),
    constraints=[{"type": "eq",
                  "fun": lambda x: float(x[0] + x[1] - 1.0),
                  "jac": lambda x: np.array([1.0, 1.0])}],
    method="SLSQP",
    options={"ftol": 1e-12},
)

x_star_qp  = result_qp_eq.x
nu_star_qp = -2.0 * x_star_qp[0]  # from stationarity: 2*x1 + nu = 0
f_star_qp  = float(x_star_qp[0] ** 2 + x_star_qp[1] ** 2)

# Dual function at nu*
nu_grid      = np.linspace(-3.0, 1.0, 300)
dual_vals_qp = -0.5 * nu_grid ** 2 - nu_grid  # analytic dual function

print("QP example: min x1^2 + x2^2  s.t.  x1 + x2 = 1")
print(f"  Primal optimum : x* = ({x_star_qp[0]:.6f}, {x_star_qp[1]:.6f})")
print(f"  Primal value   : p* = {f_star_qp:.6f}  (theory: 0.5)")
print(f"  Dual variable  : nu* = {nu_star_qp:.6f}  (theory: -1.0)")
print(f"  Dual value g(nu*) = {float(-0.5 * nu_star_qp**2 - nu_star_qp):.6f}  (theory: 0.5)")
print(f"  Duality gap    : {abs(f_star_qp - (-0.5 * nu_star_qp**2 - nu_star_qp)):.2e}")
print("")
print("Strong duality holds: p* = d* = 0.5 (verified analytically and numerically).")


### 1.6 LP Duality: Primal and Dual in Parallel

The **LP primal:** $\min_{x \geq 0}\, c^\top x$ s.t. $Ax \leq b$

The **LP dual:** $\max_{\lambda \geq 0}\, -b^\top \lambda$ s.t. $A^\top \lambda + c = 0$

We solve a simple 2-D LP using `scipy.linprog` and verify **strong duality** ($p^* = d^*$).


In [ ]:
# ── LP Primal and Dual ────────────────────────────────────────────────────────
# Primal: min -x1 - 2*x2  s.t. x1+x2<=4, x1<=2, x1>=0, x2>=0
# LP convention: minimise c^T x s.t. A_ub x <= b_ub, bounds
c_lp   = np.array([-1.0, -2.0])
A_ub   = np.array([[1.0, 1.0], [1.0, 0.0]])
b_ub   = np.array([4.0, 2.0])
bounds_lp = [(0.0, None), (0.0, None)]

result_lp = linprog(c_lp, A_ub=A_ub, b_ub=b_ub, bounds=bounds_lp, method="highs")
x_lp_star  = result_lp.x
p_star_lp  = float(result_lp.fun)

print("LP Primal: min -x1 - 2*x2  s.t.  x1+x2<=4, x1<=2, x1>=0, x2>=0")
print(f"  Primal solution: x* = ({x_lp_star[0]:.6f}, {x_lp_star[1]:.6f})")
print(f"  Primal value   : p* = {p_star_lp:.6f}  (expected -6.0)")
print("")

# LP dual: the dual variable (shadow prices) are available from the linprog result
# For scipy HiGHS, dual variables are in result_lp.ineqlin.marginals
if hasattr(result_lp, "ineqlin"):
    lam_lp = -result_lp.ineqlin.marginals  # scipy sign convention
    d_star_lp = float(-b_ub @ lam_lp)
    print(f"  Dual variables : lambda* = {lam_lp}")
    print(f"  Dual value     : d* = {d_star_lp:.6f}")
    print(f"  Duality gap    : |p* - d*| = {abs(p_star_lp - d_star_lp):.2e}")
else:
    print("  (Dual variables not available in this scipy version)")
    d_star_lp = p_star_lp  # assume strong duality for reference

# ── Visualise LP feasible region ─────────────────────────────────────────────
fig_lp, ax_lp = plt.subplots(figsize=(7, 6))

# Feasible region vertices: (0,0), (2,0), (2,2), (0,4)
feasible_verts = np.array([[0, 0], [2, 0], [2, 2], [0, 4], [0, 0]], dtype=float)
ax_lp.fill(feasible_verts[:, 0], feasible_verts[:, 1], alpha=0.20, color="steelblue",
            label="Feasible region")
ax_lp.plot(feasible_verts[:, 0], feasible_verts[:, 1], "b-", lw=1.5)

# Objective contours c^T x = const  -> -x1 - 2*x2 = k -> x2 = (-k - x1) / 2
x1_plot = np.linspace(0, 2.5, 100)
for k_val in [-2.0, -4.0, -6.0]:
    x2_contour = (-k_val - x1_plot) / 2.0
    mask_cont  = x2_contour >= 0
    ax_lp.plot(x1_plot[mask_cont], x2_contour[mask_cont], "r--", alpha=0.5, lw=1.0)
    if mask_cont.any():
        ax_lp.annotate(f"obj={k_val:.0f}",
                       xy=(x1_plot[mask_cont][0], x2_contour[mask_cont][0]),
                       fontsize=8, color="red")

ax_lp.scatter(*x_lp_star, c="tomato", s=120, zorder=10, label=f"Optimal ({x_lp_star[0]:.1f},{x_lp_star[1]:.1f})")
ax_lp.set_xlabel("$x_1$", fontsize=11)
ax_lp.set_ylabel("$x_2$", fontsize=11)
ax_lp.set_title("LP Feasible Region and Objective Contours", fontsize=11, fontweight="bold")
ax_lp.legend(fontsize=9)
ax_lp.set_xlim(-0.2, 2.8)
ax_lp.set_ylim(-0.2, 4.8)
plt.tight_layout()
plt.show()

print("Optimal at vertex (2, 2): classic LP result — optimum always at a vertex.")


### 1.6b Visualising the QP Dual Function

For the equality-constrained QP (min $x_1^2+x_2^2$ s.t. $x_1+x_2=1$), the dual function
is $g(\nu) = -\nu^2/2 - \nu$.  We plot it alongside the grid-based approximation to
confirm the analytic form is exact.


In [ ]:
# ── Visualise QP Dual Function: analytic vs grid approximation ─────────────────
nu_range   = np.linspace(-3.0, 3.0, 200)
dual_exact = -0.5 * nu_range**2 - nu_range          # g(nu) = -nu^2/2 - nu
dual_approx = np.array([
    dual_function_approx(
        lam=np.array([]), nu=np.array([float(nu_v)]),
        f=lambda x: float(x[0]**2 + x[1]**2),
        g_fns=[],
        h_fns=[lambda x: float(x[0] + x[1] - 1.0)],
        x_grid=np.c_[
            np.linspace(-2.0, 2.0, 80),
            1.0 - np.linspace(-2.0, 2.0, 80),  # x2 = 1 - x1 on constraint
        ],
    )
    for nu_v in nu_range
])

fig_dual, axes_dual = plt.subplots(1, 2, figsize=(13, 4))

# Left: dual function over nu
axes_dual[0].plot(nu_range, dual_exact, "b-", lw=2, label="Analytic g(nu)")
axes_dual[0].plot(nu_range, dual_approx, "r--", lw=1.5, label="Grid approx")
axes_dual[0].axvline(nu_star_qp, color="green", ls=":", lw=1.5, label=f"nu*={nu_star_qp:.2f}")
axes_dual[0].axhline(f_star_qp, color="orange", ls=":", lw=1.5, label=f"p*={f_star_qp:.3f}")
axes_dual[0].set_xlabel("nu (equality dual variable)", fontsize=11)
axes_dual[0].set_ylabel("g(nu) = dual value", fontsize=11)
axes_dual[0].set_title("QP Dual Function g(nu)", fontsize=11, fontweight="bold")
axes_dual[0].legend(fontsize=9)
axes_dual[0].set_ylim(-5, 1)

# Right: Lagrangian surface L(x, nu) at nu = nu*
x1_surf = np.linspace(-1.0, 2.0, 60)
x2_surf = np.linspace(-1.0, 2.0, 60)
X1s, X2s = np.meshgrid(x1_surf, x2_surf)
L_surf   = X1s**2 + X2s**2 + nu_star_qp * (X1s + X2s - 1.0)

cf = axes_dual[1].contourf(X1s, X2s, L_surf, levels=20, cmap="viridis")
plt.colorbar(cf, ax=axes_dual[1], label="Lagrangian value")
# Constraint line x1 + x2 = 1
x1_line = np.array([-0.5, 1.5])
x2_line = 1.0 - x1_line
axes_dual[1].plot(x1_line, x2_line, "r-", lw=2, label="$x_1+x_2=1$")
axes_dual[1].scatter(*x_star_qp, c="white", s=120, zorder=5, label=f"x*=({x_star_qp[0]:.2f},{x_star_qp[1]:.2f})")
axes_dual[1].set_xlabel("$x_1$", fontsize=11)
axes_dual[1].set_ylabel("$x_2$", fontsize=11)
axes_dual[1].set_title(f"Lagrangian Surface at nu*={nu_star_qp:.2f}", fontsize=11, fontweight="bold")
axes_dual[1].legend(fontsize=9)

plt.suptitle("QP Duality: Dual Function and Lagrangian Surface",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("The Lagrangian is minimised at x*=(0.5,0.5) when nu=nu* — strong duality holds.")
max_approx_err = float(np.abs(dual_exact - dual_approx).max())
print(f"Max analytic vs grid-approx error: {max_approx_err:.4f}")


---
### 1.7 KKT Conditions

For **min** $f(x)$ s.t. $g_i(x) \leq 0$, $h_j(x) = 0$, the four KKT conditions are:

| Condition | Equation | Intuition |
|-----------|----------|-----------|
| **Stationarity** | $\nabla f + \sum_i \lambda_i \nabla g_i + \sum_j \nu_j \nabla h_j = 0$ | Gradient of Lagrangian is zero |
| **Primal feasibility** | $g_i(x^*) \leq 0$, $h_j(x^*) = 0$ | Constraints are satisfied |
| **Dual feasibility** | $\lambda_i \geq 0$ | Inequality multipliers are non-negative |
| **Comp. slackness** | $\lambda_i g_i(x^*) = 0$ | Either constraint is active OR multiplier is zero |

For **convex** problems with Slater's condition: KKT is **necessary and sufficient** for optimality.


In [ ]:
# ── Part 1: KKT Conditions Checker ────────────────────────────────────────────


def check_kkt_conditions(
    grad_f_at_x: np.ndarray,
    g_vals: np.ndarray,
    grad_g_at_x: np.ndarray,
    h_vals: np.ndarray,
    grad_h_at_x: np.ndarray,
    lam: np.ndarray,
    nu: np.ndarray,
    tol: float = 1e-5,
) -> dict[str, Any]:
    '''Verify the four KKT conditions at a candidate primal-dual solution.

    Conditions for min f(x) s.t. g_i(x) <= 0, h_j(x) = 0:
      1. Stationarity         : grad_f + lam @ grad_g + nu @ grad_h = 0
      2. Primal ineq. feasibility: g_i(x*) <= 0
      3. Primal eq. feasibility  : h_j(x*) = 0
      4. Dual feasibility        : lam_i >= 0
      5. Complementary slackness : lam_i * g_i(x*) = 0

    Args:
        grad_f_at_x: Gradient of f at x*, shape (n_vars,).
        g_vals: Inequality constraint values g_i(x*), shape (n_ineq,).
        grad_g_at_x: Jacobian of inequality constraints, shape (n_ineq, n_vars).
        h_vals: Equality constraint values h_j(x*), shape (n_eq,).
        grad_h_at_x: Jacobian of equality constraints, shape (n_eq, n_vars).
        lam: Inequality dual variables lambda >= 0, shape (n_ineq,).
        nu: Equality dual variables nu, shape (n_eq,).
        tol: Numerical tolerance for approximate equality checks.

    Returns:
        Dict with per-condition boolean flags and residual norms.
    '''
    # 1. Stationarity
    stat_vec = grad_f_at_x.copy().astype(float)
    if lam.size > 0 and grad_g_at_x.size > 0:
        stat_vec = stat_vec + lam @ grad_g_at_x
    if nu.size > 0 and grad_h_at_x.size > 0:
        stat_vec = stat_vec + nu @ grad_h_at_x
    stat_norm = float(np.linalg.norm(stat_vec))
    stat_ok   = stat_norm <= tol

    # 2 & 3. Primal feasibility
    prim_ineq_ok = bool((g_vals <= tol).all()) if g_vals.size > 0 else True
    prim_eq_ok   = bool((np.abs(h_vals) <= tol).all()) if h_vals.size > 0 else True

    # 4. Dual feasibility
    dual_ok = bool((lam >= -tol).all()) if lam.size > 0 else True

    # 5. Complementary slackness
    cs_vals  = lam * g_vals if lam.size > 0 else np.array([])
    cs_norm  = float(np.linalg.norm(cs_vals)) if cs_vals.size > 0 else 0.0
    cs_ok    = cs_norm <= tol

    all_ok = stat_ok and prim_ineq_ok and prim_eq_ok and dual_ok and cs_ok

    return {
        "stationarity":             stat_ok,
        "primal_ineq_feasible":     prim_ineq_ok,
        "primal_eq_feasible":       prim_eq_ok,
        "dual_feasible":            dual_ok,
        "complementary_slackness":  cs_ok,
        "all_satisfied":            all_ok,
        "stationarity_norm":        round(stat_norm, 8),
        "comp_slackness_norm":      round(cs_norm, 8),
    }


# ── Verify KKT at QP solution x* = (0.5, 0.5), nu* = -1 ─────────────────────
# min x1^2 + x2^2  s.t. x1+x2-1 = 0
x_qp    = x_star_qp
nu_qp   = np.array([nu_star_qp])
grad_f  = np.array([2.0 * x_qp[0], 2.0 * x_qp[1]])
h_val   = np.array([x_qp[0] + x_qp[1] - 1.0])
grad_h  = np.array([[1.0, 1.0]])

kkt_qp = check_kkt_conditions(
    grad_f_at_x=grad_f,
    g_vals=np.array([]), grad_g_at_x=np.empty((0, 2)),
    h_vals=h_val,         grad_h_at_x=grad_h,
    lam=np.array([]),     nu=nu_qp,
    tol=KKT_TOL,
)

print("KKT verification at QP optimum (x*=(0.5,0.5), nu*=-1.0):")
for condition, satisfied in kkt_qp.items():
    if isinstance(satisfied, bool):
        status = "PASS" if satisfied else "FAIL"
        print(f"  {condition:<28s}: {status}")
    else:
        print(f"  {condition:<28s}: {satisfied:.2e}")
print(f"  All KKT conditions satisfied  : {'YES' if kkt_qp['all_satisfied'] else 'NO'}")


### 1.8 Why Convexity Matters for Gradient Descent

Gradient descent is guaranteed to converge to the **global minimum** on convex objectives (for small enough step size). On non-convex objectives, the algorithm gets trapped in local minima — outcome depends on initialisation.


In [ ]:
# ── Gradient Descent: Convergence on Convex vs Non-Convex Objectives ──────────
# Convex problems: GD converges to global minimum for any starting point.
# Non-convex: GD may get stuck in a local minimum depending on initialisation.


def gradient_descent_1d(
    f: "Callable[[float], float]",
    grad_f: "Callable[[float], float]",
    x0: float,
    lr: float = 0.05,
    n_steps: int = 80,
) -> tuple[list[float], list[float]]:
    '''Run gradient descent on a 1-D objective.

    Args:
        f: Objective function.
        grad_f: Gradient (derivative) of f.
        x0: Initial point.
        lr: Step size (learning rate).
        n_steps: Number of gradient steps.

    Returns:
        Tuple (x_history, f_history) of iterates and function values.
    '''
    x = float(x0)
    x_hist: list[float] = [x]
    f_hist: list[float] = [float(f(x))]
    for _ in range(n_steps):
        x = x - lr * float(grad_f(x))
        x_hist.append(x)
        f_hist.append(float(f(x)))
    return x_hist, f_hist


# ── Convex case: f(x) = x^2  (global min at x=0) ────────────────────────────
f_cvx_gd     = lambda x: x ** 2
grad_cvx_gd  = lambda x: 2.0 * x

starts_cvx = [-3.0, -1.5, 0.5, 2.0]
results_cvx: list = []
for x0_c in starts_cvx:
    xh, fh = gradient_descent_1d(f_cvx_gd, grad_cvx_gd, x0_c, lr=0.1, n_steps=60)
    results_cvx.append((x0_c, xh, fh))

# ── Non-convex case: f(x) = x^4 - 3x^2 + x  (local minima at +-1.2) ─────────
f_nc_gd     = lambda x: x**4 - 3.0 * x**2 + x
grad_nc_gd  = lambda x: 4.0 * x**3 - 6.0 * x + 1.0

starts_nc = [-2.0, -0.5, 0.3, 1.8]
results_nc: list = []
for x0_n in starts_nc:
    xh, fh = gradient_descent_1d(f_nc_gd, grad_nc_gd, x0_n, lr=0.02, n_steps=100)
    results_nc.append((x0_n, xh, fh))

# ── Visualise ─────────────────────────────────────────────────────────────────
fig_gd, axes_gd = plt.subplots(1, 2, figsize=(14, 5))
x_vis = np.linspace(-3.2, 3.2, 400)

# Convex
axes_gd[0].plot(x_vis, f_cvx_gd(x_vis), "k-", lw=2, label="f(x) = x^2")
colors_gd = ["steelblue", "green", "orange", "purple"]
for (x0_c, xh, fh), col_g in zip(results_cvx, colors_gd):
    axes_gd[0].plot(xh, [f_cvx_gd(x) for x in xh], "o-",
                    color=col_g, markersize=3, lw=1.2, alpha=0.8,
                    label=f"x0={x0_c:.1f} -> {xh[-1]:.4f}")
axes_gd[0].axvline(0, color="red", ls="--", lw=1.5, label="Global min x*=0")
axes_gd[0].set_xlabel("x", fontsize=11)
axes_gd[0].set_ylabel("f(x)", fontsize=11)
axes_gd[0].set_title("Convex: GD always finds global min", fontsize=11, fontweight="bold")
axes_gd[0].legend(fontsize=8)
axes_gd[0].set_xlim(-3.5, 3.5)
axes_gd[0].set_ylim(-0.5, 10)

# Non-convex
axes_gd[1].plot(x_vis, f_nc_gd(x_vis), "k-", lw=2, label="f(x) = x^4 - 3x^2 + x")
x_minima = [-1.28, 1.22]
for xm in x_minima:
    axes_gd[1].axvline(xm, color="gray", ls=":", lw=1.2)
    axes_gd[1].annotate(f"local min\nx={xm:.2f}", xy=(xm, f_nc_gd(xm)),
                        xytext=(xm + 0.4, f_nc_gd(xm) + 1.5),
                        arrowprops=dict(arrowstyle="->", color="gray"),
                        fontsize=8, color="gray")
for (x0_n, xh, fh), col_g in zip(results_nc, colors_gd):
    axes_gd[1].plot(xh, [f_nc_gd(x) for x in xh], "o-",
                    color=col_g, markersize=3, lw=1.2, alpha=0.8,
                    label=f"x0={x0_n:.1f} -> {xh[-1]:.3f}")
axes_gd[1].set_xlabel("x", fontsize=11)
axes_gd[1].set_ylabel("f(x)", fontsize=11)
axes_gd[1].set_title("Non-Convex: GD stuck at different local minima", fontsize=11, fontweight="bold")
axes_gd[1].legend(fontsize=8)
axes_gd[1].set_xlim(-2.5, 2.5)
axes_gd[1].set_ylim(-5, 5)

plt.suptitle("Why Convexity Matters for Optimisation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Print final values
print("GD convergence summary:")
print(f"  Convex f(x)=x^2: all runs converge to x~0 regardless of start.")
for x0_c, xh, fh in results_cvx:
    print(f"    x0={x0_c:+5.1f} -> final x={xh[-1]:.5f}  (|x*-0|={abs(xh[-1]):.5f})")
print(f"  Non-convex: runs converge to DIFFERENT local minima depending on start.")
for x0_n, xh, fh in results_nc:
    print(f"    x0={x0_n:+5.1f} -> final x={xh[-1]:.5f}  f={fh[-1]:.5f}")


---
## Part 2 — Empirical Validation

We now run two experiments that validate the key duality theorems:

1. **Strong duality sweep:** solve the LP and QP at multiple parameter settings and
   confirm the duality gap is numerically zero (within solver tolerance).
2. **Complementary slackness visualisation:** show which LP constraints are active at the
   optimum and verify $\lambda_i g_i(x^*) = 0$ for each.


In [ ]:
# ── Strong Duality Verification: LP at Multiple RHS Settings ──────────────────
print("Strong duality verification: vary LP right-hand-side b and check |p* - d*|")
print(f"{'b1':>6s}  {'b2':>6s}  {'Primal p*':>10s}  {'Dual d*':>10s}  {'Gap':>12s}")
print("-" * 52)

b1_vals = [3.0, 4.0, 5.0, 6.0]
b2_vals = [1.5, 2.0, 2.5, 3.0]

for b1_v, b2_v in zip(b1_vals, b2_vals):
    b_lp_v    = np.array([b1_v, b2_v])
    res_prim  = linprog(c_lp, A_ub=A_ub, b_ub=b_lp_v, bounds=bounds_lp, method="highs")
    p_star_v  = float(res_prim.fun)
    if hasattr(res_prim, "ineqlin"):
        lam_v    = -res_prim.ineqlin.marginals
        d_star_v = float(-b_lp_v @ lam_v)
        gap_v    = abs(p_star_v - d_star_v)
    else:
        d_star_v = p_star_v
        gap_v    = 0.0
    print(f"  {b1_v:4.1f}  {b2_v:4.1f}  {p_star_v:10.5f}  {d_star_v:10.5f}  {gap_v:12.2e}")

print("")
print("Duality gap is numerically zero for all LP instances — strong duality confirmed.")

# ── Complementary slackness at LP optimum ────────────────────────────────────
print("")
print("Complementary slackness at LP optimum x* = (2.0, 2.0):")
g1_val = float(x_lp_star[0] + x_lp_star[1] - 4.0)   # x1+x2 <= 4  -> active (=0)
g2_val = float(x_lp_star[0] - 2.0)                   # x1 <= 2     -> active (=0)
g3_val = float(-x_lp_star[0])                        # x1 >= 0     -> inactive
g4_val = float(-x_lp_star[1])                        # x2 >= 0     -> inactive
g_all  = np.array([g1_val, g2_val, g3_val, g4_val])

if hasattr(result_lp, "ineqlin"):
    lam_ext = np.concatenate([lam_lp, -result_lp.lower.marginals])
else:
    lam_ext = np.zeros(4)

constraint_names = ["x1+x2<=4 (active)", "x1<=2   (active)",
                    "x1>=0   (inactive)", "x2>=0   (inactive)"]
for cname, g_v in zip(constraint_names, g_all):
    status = "ACTIVE" if abs(g_v) < 1e-4 else "inactive"
    print(f"  {cname}: g_i(x*) = {g_v:+.5f}  [{status}]")
print("")
print("Active constraints (g_i = 0): lambda_i can be nonzero (forces are needed).")
print("Inactive constraints (g_i < 0): lambda_i must be 0 (complementary slackness).")


---
## Part 3 — Application: SVM Dual Derivation and Solution

### 3.1 From SVM Primal to Dual via Lagrangian

The soft-margin SVM primal:
$$
\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_i \xi_i
\quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i,\; \xi_i \geq 0
$$

The Lagrangian (with $\alpha_i \geq 0$ for margin constraints, $\mu_i \geq 0$ for $\xi_i \geq 0$):
$$
L = \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_i\xi_i
  - \sum_i \alpha_i(y_i(\mathbf{w}^\top \mathbf{x}_i+b) - 1 + \xi_i)
  - \sum_i \mu_i \xi_i
$$

**KKT stationarity** gives:
$$
\frac{\partial L}{\partial \mathbf{w}} = 0 \Rightarrow \mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i
$$
$$
\frac{\partial L}{\partial b} = 0 \Rightarrow \sum_i \alpha_i y_i = 0
$$
$$
\frac{\partial L}{\partial \xi_i} = 0 \Rightarrow \mu_i = C - \alpha_i
$$

Substituting back yields the **SVM dual**:
$$
\max_{\boldsymbol{\alpha}} \sum_i \alpha_i - \frac{1}{2}\sum_{ij} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^\top \mathbf{x}_j
\quad \text{s.t.} \quad 0 \leq \alpha_i \leq C, \; \sum_i \alpha_i y_i = 0
$$

**Key insight:** the objective depends only on $\mathbf{x}_i^\top \mathbf{x}_j$ — replace with $K(\mathbf{x}_i, \mathbf{x}_j)$ for the kernel trick!


### 3.2 Implementing the SVM Dual

We solve the dual QP with `scipy.optimize.minimize` (SLSQP method), minimising
the *negative* dual objective (since we want to maximise).


In [ ]:
# ── Part 3: SVM Dual Implementation ───────────────────────────────────────────


def soft_margin_svm_dual(
    X: np.ndarray,
    y: np.ndarray,
    C: float,
    kernel_matrix: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    '''Solve the soft-margin SVM dual QP via scipy.optimize.minimize (SLSQP).

    Maximises the dual objective:
        W(alpha) = sum(alpha) - 0.5 * alpha^T K_y alpha
    where K_y[i,j] = y_i * y_j * K(x_i, x_j).

    Subject to:
        0 <= alpha_i <= C     (box constraint, dual feasibility + primal ineq.)
        sum(alpha_i * y_i) = 0  (KKT stationarity w.r.t. b)

    Args:
        X: Feature matrix, shape (n_train, n_features).
        y: Binary labels in {-1, +1}, shape (n_train,).
        C: Soft-margin regularisation parameter.
        kernel_matrix: Optional pre-computed Gram matrix (n_train, n_train).
            Defaults to linear kernel K = X @ X.T.

    Returns:
        Tuple (alphas, margin_sv_mask, bias, w):
            alphas: Dual variables, shape (n_train,).
            margin_sv_mask: Boolean mask for margin support vectors (0 < alpha < C).
            bias: Scalar intercept b.
            w: Primal weight vector from w = sum(alpha_i * y_i * x_i), shape (n_features,).
    '''
    n_tr  = len(y)
    K     = X @ X.T if kernel_matrix is None else kernel_matrix
    K_y   = (y[:, None] * y[None, :]) * K  # K_y[i,j] = y_i * y_j * K_ij

    # Objective: minimise NEGATIVE dual (scipy minimises)
    neg_dual      = lambda a: float(0.5 * a @ K_y @ a - a.sum())
    neg_dual_grad = lambda a: (K_y @ a - np.ones(n_tr))

    alpha0      = np.zeros(n_tr)
    bounds_svm  = [(0.0, float(C))] * n_tr
    constraints = [{"type": "eq",
                    "fun": lambda a, _y=y: float((_y * a).sum()),
                    "jac": lambda a, _y=y: _y.astype(float)}]

    result_svm = scipy_minimize(
        neg_dual, alpha0, jac=neg_dual_grad,
        method="SLSQP", bounds=bounds_svm, constraints=constraints,
        options={"maxiter": 2000, "ftol": 1e-10},
    )

    alphas = np.clip(result_svm.x, 0.0, float(C))
    sv_tol         = DUAL_TOL
    sv_mask        = alphas > sv_tol                       # all SVs (alpha > 0)
    margin_sv_mask = (alphas > sv_tol) & (alphas < C - sv_tol)  # margin SVs

    w = (alphas * y) @ X  # primal weight vector from stationarity condition
    if margin_sv_mask.any():
        bias = float(np.mean(y[margin_sv_mask] - X[margin_sv_mask] @ w))
    elif sv_mask.any():
        bias = float(np.mean(y[sv_mask] - X[sv_mask] @ w))
    else:
        bias = 0.0

    return alphas, margin_sv_mask, bias, w


print("soft_margin_svm_dual defined.")
print(f"  Solver : scipy SLSQP   |  C = {C_SVM}")
print(f"  Dual variables: one per training point (alpha_i in [0, C])")
print("  Support vectors: training points with alpha_i > 0")
print("  Margin SVs: points with 0 < alpha_i < C (on the margin boundary)")


In [ ]:
# ── Solve SVM Dual on Classification Data ─────────────────────────────────────
print(f"Running SVM dual on {X_tr_svm.shape[0]} training points (C={C_SVM}) ...")

alphas_svm, margin_sv_mask, bias_svm, w_svm = soft_margin_svm_dual(
    X_tr_svm, y_tr_svm, C=C_SVM,
)

n_sv_total  = int((alphas_svm > DUAL_TOL).sum())
n_sv_margin = int(margin_sv_mask.sum())
dual_obj    = float(alphas_svm.sum()
               - 0.5 * alphas_svm @ ((y_tr_svm[:, None] * y_tr_svm[None, :])
                                     * (X_tr_svm @ X_tr_svm.T)) @ alphas_svm)

print(f"  Total support vectors   : {n_sv_total} / {len(y_tr_svm)}"
      f" ({100*n_sv_total/len(y_tr_svm):.1f}%)")
print(f"  Margin support vectors  : {n_sv_margin}")
print(f"  Dual objective W(alpha*): {dual_obj:.5f}")
print(f"  ||w||_2                 : {float(np.linalg.norm(w_svm)):.5f}")
print(f"  Bias b                  : {bias_svm:.5f}")
print("")

# Primal constraint check: y_i * (w^T x_i + b) >= 1 - xi_i
margins_tr = y_tr_svm * (X_tr_svm @ w_svm + bias_svm)
xi_svm     = np.maximum(0.0, 1.0 - margins_tr)   # slack variables
primal_obj = 0.5 * float(np.linalg.norm(w_svm) ** 2) + C_SVM * float(xi_svm.sum())
print(f"  Primal objective (1/2||w||^2 + C*sum(xi)): {primal_obj:.5f}")
print(f"  Duality gap (primal - dual)               : {abs(primal_obj - dual_obj):.2e}")
print("")
print(f"  {'alpha_i':>10s}  {'y_i':>4s}  {'margin':>8s}  {'Type'}")
print("  " + "-" * 36)
for i in np.where(alphas_svm > DUAL_TOL)[0][:10]:
    sv_type = "margin SV" if margin_sv_mask[i] else "bound SV"
    print(f"  {alphas_svm[i]:10.5f}  {int(y_tr_svm[i]):+4d}  {margins_tr[i]:8.5f}  {sv_type}")
if n_sv_total > 10:
    print(f"  ... ({n_sv_total - 10} more support vectors)")


### 3.3 Verifying KKT Conditions at the SVM Optimum

At the SVM optimum, KKT conditions reduce to:
- **Stationarity** $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$: residual $\|\mathbf{w} - \sum_i \alpha_i y_i \mathbf{x}_i\|$ should be $\approx 0$.
- **Constraint** $\sum_i \alpha_i y_i = 0$: residual should be $\approx 0$.
- **Complementary slackness** $\alpha_i (y_i(\mathbf{w}^\top \mathbf{x}_i + b) - 1 + \xi_i) = 0$.


In [ ]:
# ── KKT Conditions at SVM Optimum ─────────────────────────────────────────────
# Stationarity: w = sum_i alpha_i * y_i * x_i
w_from_dual   = (alphas_svm * y_tr_svm) @ X_tr_svm
stat_residual = float(np.linalg.norm(w_svm - w_from_dual))

# Zero-sum: sum(alpha_i * y_i) = 0
sum_alpha_y = float((alphas_svm * y_tr_svm).sum())

# Complementary slackness: alpha_i * (y_i * (w^T x_i + b) - 1 + xi_i) = 0
cs_svm = alphas_svm * (margins_tr - 1.0 + xi_svm)
cs_norm_svm = float(np.linalg.norm(cs_svm))

# Primal feasibility: y_i*(w^T x_i + b) >= 1 - xi_i  <=> margins_i >= 1 - xi_i
prim_feasible = bool(np.all(margins_tr >= 1.0 - xi_svm - KKT_TOL))

# Dual feasibility: 0 <= alpha_i <= C
dual_feasible = bool(np.all(alphas_svm >= -KKT_TOL) and np.all(alphas_svm <= C_SVM + KKT_TOL))

print("KKT Conditions at SVM Optimum:")
print(f"  Stationarity (||w - sum(alpha_i*y_i*x_i)||): {stat_residual:.2e}"
      f"  {'PASS' if stat_residual < KKT_TOL else 'FAIL'}")
print(f"  Zero-sum (|sum(alpha_i * y_i)|)            : {abs(sum_alpha_y):.2e}"
      f"  {'PASS' if abs(sum_alpha_y) < KKT_TOL else 'FAIL'}")
print(f"  Complementary slackness norm               : {cs_norm_svm:.2e}"
      f"  {'PASS' if cs_norm_svm < 0.1 else 'FAIL'}")
print(f"  Primal feasibility                         : {'PASS' if prim_feasible else 'FAIL'}")
print(f"  Dual feasibility (0 <= alpha <= C)         : {'PASS' if dual_feasible else 'FAIL'}")
print("")
print("All KKT conditions are satisfied at the dual optimum — verifying that")
print("the scipy solver converged to a true primal-dual optimal pair.")


### 3.4 Effect of the Regularisation Parameter C

The soft-margin SVM trades off margin width against constraint violations via C:
- **Large C:** hard margin — favour correct classification, many support vectors, narrow margin.
- **Small C:** soft margin — tolerate more violations, fewer support vectors, wider margin.


In [ ]:
# ── SVM Dual: Sweep C and Record Statistics ────────────────────────────────────
C_values_sweep = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]
sweep_rows: list = []

print(f"{'C':>8s}  {'#SVs':>6s}  {'#MarginSVs':>11s}  {'||w||':>8s}  {'TrainAcc':>9s}  {'TestAcc':>8s}  {'DualObj':>9s}")
print("-" * 72)

for c_val in C_values_sweep:
    a_c, msv_c, b_c, w_c = soft_margin_svm_dual(X_tr_svm, y_tr_svm, C=c_val)
    n_sv_c      = int((a_c > DUAL_TOL).sum())
    n_msv_c     = int(msv_c.sum())
    norm_w_c    = float(np.linalg.norm(w_c))
    y_pred_tr_c = np.sign(X_tr_svm @ w_c + b_c).astype(int)
    y_pred_te_c = np.sign(X_te_svm @ w_c + b_c).astype(int)
    acc_tr_c    = float(np.mean(y_pred_tr_c == y_tr_svm))
    acc_te_c    = float(np.mean(y_pred_te_c == y_te_svm))
    K_y_c       = (y_tr_svm[:, None] * y_tr_svm[None, :]) * (X_tr_svm @ X_tr_svm.T)
    dual_obj_c  = float(a_c.sum() - 0.5 * a_c @ K_y_c @ a_c)
    sweep_rows.append((c_val, n_sv_c, n_msv_c, norm_w_c, acc_tr_c, acc_te_c, dual_obj_c))
    print(f"  {c_val:6.2f}  {n_sv_c:6d}  {n_msv_c:11d}  {norm_w_c:8.4f}  {acc_tr_c:9.4f}  {acc_te_c:8.4f}  {dual_obj_c:9.4f}")

print("")
print("Interpretation:")
print("  Small C -> soft margin: fewer SVs but wider margin (smaller ||w||).")
print("  Large C -> hard margin: more SVs, narrower margin (larger ||w||).")
print("  Test accuracy peaks at intermediate C (bias-variance tradeoff).")

# Visualise C sweep
C_arr     = np.array([r[0] for r in sweep_rows])
n_sv_arr  = np.array([r[1] for r in sweep_rows])
norm_arr  = np.array([r[3] for r in sweep_rows])
acc_te_arr = np.array([r[5] for r in sweep_rows])

fig_c, axes_c = plt.subplots(1, 3, figsize=(15, 4))

axes_c[0].semilogx(C_arr, n_sv_arr, "bo-", lw=2, markersize=7)
axes_c[0].set_xlabel("C (log scale)", fontsize=11)
axes_c[0].set_ylabel("Number of Support Vectors", fontsize=11)
axes_c[0].set_title("SVs vs C", fontsize=11, fontweight="bold")
axes_c[0].grid(True, alpha=0.3)

axes_c[1].semilogx(C_arr, norm_arr, "rs-", lw=2, markersize=7)
axes_c[1].set_xlabel("C (log scale)", fontsize=11)
axes_c[1].set_ylabel("||w||_2 (inverse margin width)", fontsize=11)
axes_c[1].set_title("Margin Width vs C", fontsize=11, fontweight="bold")
axes_c[1].grid(True, alpha=0.3)

axes_c[2].semilogx(C_arr, acc_te_arr, "gD-", lw=2, markersize=7)
axes_c[2].set_xlabel("C (log scale)", fontsize=11)
axes_c[2].set_ylabel("Test Accuracy", fontsize=11)
axes_c[2].set_title("Test Accuracy vs C", fontsize=11, fontweight="bold")
axes_c[2].set_ylim(0.7, 1.05)
axes_c[2].grid(True, alpha=0.3)

plt.suptitle("SVM Regularisation: Effect of C on Margin, SVs, and Accuracy",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Best C from sweep:", C_arr[int(np.argmax(acc_te_arr))])


In [ ]:
# ── Compare Dual SVM vs sklearn SVC ──────────────────────────────────────────
sklearn_svc = SVC(kernel="linear", C=C_SVM)
sklearn_svc.fit(X_tr_svm, y_tr_svm)

# Predictions
y_pred_dual   = np.sign(X_te_svm @ w_svm + bias_svm).astype(int)
y_pred_sklearn = sklearn_svc.predict(X_te_svm)

acc_dual   = float(np.mean(y_pred_dual    == y_te_svm))
acc_sklearn = float(np.mean(y_pred_sklearn == y_te_svm))

print("Comparing our SVM dual implementation vs sklearn SVC (linear kernel):")
print(f"  Our dual SVM accuracy  : {acc_dual:.4f}")
print(f"  sklearn SVC accuracy   : {acc_sklearn:.4f}")
print(f"  Accuracy difference    : {abs(acc_dual - acc_sklearn):.4f}")
print("")
print(f"  Our ||w||   : {float(np.linalg.norm(w_svm)):.5f}")
print(f"  sklearn ||w|: {float(np.linalg.norm(sklearn_svc.coef_[0])):.5f}")
print(f"  Our bias b  : {bias_svm:.5f}")
print(f"  sklearn bias: {float(sklearn_svc.intercept_[0]):.5f}")
print("")

# Decision boundary visualisation
fig_svm, axes_svm = plt.subplots(1, 2, figsize=(14, 5))
for ax_sv, (title_sv, w_sv, b_sv, n_sv) in zip(
    axes_svm,
    [("Our Dual SVM", w_svm, bias_svm, n_sv_total),
     ("sklearn SVC (reference)", sklearn_svc.coef_[0], float(sklearn_svc.intercept_[0]),
      int(sklearn_svc.support_.shape[0]))],
):
    x_min_sv = X_te_svm[:, 0].min() - 0.5
    x_max_sv = X_te_svm[:, 0].max() + 0.5
    y_min_sv = X_te_svm[:, 1].min() - 0.5
    y_max_sv = X_te_svm[:, 1].max() + 0.5
    xx_sv, yy_sv = np.meshgrid(
        np.linspace(x_min_sv, x_max_sv, 150),
        np.linspace(y_min_sv, y_max_sv, 150),
    )
    Z_sv = (np.c_[xx_sv.ravel(), yy_sv.ravel()] @ w_sv + b_sv).reshape(xx_sv.shape)
    ax_sv.contourf(xx_sv, yy_sv, Z_sv, levels=0, alpha=0.25, cmap="RdBu")
    ax_sv.contour(xx_sv, yy_sv, Z_sv, levels=[-1, 0, 1],
                  linestyles=["--", "-", "--"], colors=["gray", "black", "gray"])
    ax_sv.scatter(X_te_svm[:, 0], X_te_svm[:, 1],
                  c=y_te_svm, cmap="RdBu", edgecolors="k", s=20, lw=0.4)
    ax_sv.set_title(f"{title_sv} | {n_sv} SVs | acc={acc_dual:.3f}",
                    fontsize=10, fontweight="bold")
    ax_sv.set_xlabel("Feature 0")
    ax_sv.set_ylabel("Feature 1")

plt.suptitle("SVM Decision Boundaries: Dual vs sklearn (dashed = margin)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Solid line = decision boundary (f(x)=0). Dashed = margin planes (f(x)=+-1).")


### 3.5 Support Vector Analysis

The dual variables $\\alpha_i$ partition training points into three types:
- $\\alpha_i = 0$: non-support vector (correctly classified, away from margin).
- $0 < \\alpha_i < C$: **margin support vector** — exactly on the margin hyperplane.
- $\\alpha_i = C$: **bound support vector** — inside or on wrong side of margin (slack $\\xi_i > 0$).


In [ ]:
# ── Support Vector Analysis: Alpha Distribution and Margin Geometry ────────────
sv_mask_all = alphas_svm > DUAL_TOL
sv_alphas   = alphas_svm[sv_mask_all]
sv_margins  = margins_tr[sv_mask_all]
sv_labels   = y_tr_svm[sv_mask_all]
n_pos_sv    = int((sv_labels == 1).sum())
n_neg_sv    = int((sv_labels == -1).sum())
margin_svs  = sv_alphas[margin_sv_mask[sv_mask_all]]   # 0 < alpha < C
bound_svs   = sv_alphas[~margin_sv_mask[sv_mask_all]]  # alpha = C

print(f"Support vector breakdown (C={C_SVM}):")
print(f"  Total SVs         : {n_sv_total} ({n_sv_total/len(y_tr_svm)*100:.1f}% of train set)")
print(f"  Margin SVs (0<a<C): {n_sv_margin} — on the margin hyperplane")
print(f"  Bound SVs  (a=C)  : {n_sv_total - n_sv_margin} — inside or wrong side of margin")
print(f"  Positive-class SVs: {n_pos_sv}   Negative-class SVs: {n_neg_sv}")

fig_sv, axes_sv = plt.subplots(1, 2, figsize=(13, 4))

# Alpha histogram
axes_sv[0].hist(alphas_svm[alphas_svm > DUAL_TOL], bins=25, color="steelblue",
                edgecolor="white", lw=0.5)
axes_sv[0].axvline(C_SVM, color="red", ls="--", lw=2, label=f"C={C_SVM}")
axes_sv[0].set_xlabel("alpha_i", fontsize=11)
axes_sv[0].set_ylabel("Count", fontsize=11)
axes_sv[0].set_title("Dual Variable Distribution (alpha > 0)", fontsize=11, fontweight="bold")
axes_sv[0].legend(fontsize=9)
axes_sv[0].text(0.02, 0.95, f"Margin SVs: {n_sv_margin} | Bound SVs: {n_sv_total-n_sv_margin}",
                transform=axes_sv[0].transAxes, fontsize=9, va="top")

# Margin scatter: functional margin y*(w^T x + b)
non_sv_mask = alphas_svm <= DUAL_TOL
axes_sv[1].scatter(
    np.where(non_sv_mask)[0], margins_tr[non_sv_mask],
    c="lightgray", s=12, label="Non-SVs", alpha=0.6,
)
axes_sv[1].scatter(
    np.where(margin_sv_mask)[0], margins_tr[margin_sv_mask],
    c="green", s=40, zorder=5, label=f"Margin SVs (alpha<C)",
)
axes_sv[1].scatter(
    np.where(sv_mask_all & ~margin_sv_mask)[0],
    margins_tr[sv_mask_all & ~margin_sv_mask],
    c="red", s=40, marker="^", zorder=5, label=f"Bound SVs (alpha=C)",
)
axes_sv[1].axhline(1.0, color="black", ls="--", lw=1.5, label="Margin boundary y*(wx+b)=1")
axes_sv[1].axhline(0.0, color="gray", ls=":", lw=1.2, label="Decision boundary")
axes_sv[1].set_xlabel("Training point index", fontsize=11)
axes_sv[1].set_ylabel("Functional margin y_i*(w^T x_i + b)", fontsize=11)
axes_sv[1].set_title("Functional Margins by SV Type", fontsize=11, fontweight="bold")
axes_sv[1].legend(fontsize=8, loc="upper right")

plt.suptitle("SVM Support Vector Analysis", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Margin SVs sit exactly on the margin (functional margin ≈ 1.0).")
print("Bound SVs violate the margin (functional margin < 1.0) — slack xi > 0.")
print(f"  Mean margin for margin SVs: {float(margins_tr[margin_sv_mask].mean()):.4f} (should be ~1.0)")


---
## Part 4 — Theory vs Practice: Slater's Condition and Duality Gaps

### 4.1 Slater's Condition

**Slater's condition** (for convex problems): strong duality holds if there exists
a strictly feasible point $\tilde{x}$ such that $g_i(\tilde{x}) < 0$ for all $i$.

Without Slater's condition:
- The problem may have a **positive duality gap** $p^* > d^*$.
- KKT conditions may still hold but are no longer sufficient.

**Practical implication:** before applying dual methods (SVMs, constrained RL), verify
that the problem is convex and that a strictly feasible point exists.


In [ ]:
# ── Slater's Condition Demonstration ──────────────────────────────────────────


def check_slaters_condition(
    x_candidate: np.ndarray,
    g_fns: list[Callable[[np.ndarray], float]],
    h_fns: list[Callable[[np.ndarray], float]],
    strict_tol: float = 1e-6,
) -> dict[str, Any]:
    '''Verify Slater's condition at a candidate point.

    Slater's condition: a strictly feasible point exists — i.e., there exists
    x such that g_i(x) < 0 for all inequality constraints and h_j(x) = 0 for
    equality constraints.

    Args:
        x_candidate: Candidate point to test for strict feasibility, shape (n_vars,).
        g_fns: List of inequality constraint functions g_i (should satisfy g_i < 0).
        h_fns: List of equality constraint functions h_j (should satisfy h_j = 0).
        strict_tol: Value below which a constraint is considered strictly satisfied.

    Returns:
        Dict with keys: slater_satisfied, g_values, h_values, min_g_margin.
    '''
    g_vals = np.array([float(g(x_candidate)) for g in g_fns]) if g_fns else np.array([])
    h_vals = np.array([float(h(x_candidate)) for h in h_fns]) if h_fns else np.array([])
    ineq_strict = bool((g_vals < -strict_tol).all()) if g_vals.size > 0 else True
    eq_exact    = bool((np.abs(h_vals) < strict_tol).all()) if h_vals.size > 0 else True
    min_margin  = float(g_vals.min()) if g_vals.size > 0 else 0.0
    return {
        "slater_satisfied": ineq_strict and eq_exact,
        "g_values":         g_vals,
        "h_values":         h_vals,
        "min_g_margin":     min_margin,
    }


# ── LP example: Slater's condition holds ─────────────────────────────────────
# Strictly feasible point for LP: x1 + x2 < 4, x1 < 2, x1 > 0, x2 > 0
x_slater_lp = np.array([1.0, 1.0])
g_fns_lp    = [
    lambda x: float(x[0] + x[1] - 4.0),  # x1+x2 <= 4
    lambda x: float(x[0] - 2.0),          # x1 <= 2
    lambda x: float(-x[0]),               # x1 >= 0
    lambda x: float(-x[1]),               # x2 >= 0
]

slater_lp = check_slaters_condition(x_slater_lp, g_fns_lp, [])
print("Slater's condition for LP at x = (1.0, 1.0):")
print(f"  Satisfied: {slater_lp['slater_satisfied']}")
print(f"  g_i(x)   : {slater_lp['g_values'].round(4)}")
print(f"  min margin: {slater_lp['min_g_margin']:.4f}  (must be < 0 for strict feasibility)")
print("")

# ── Example where strong duality fails: non-convex problem ───────────────────
# min x^2  s.t. x^2 <= -1  (infeasible -> duality gap = infinity conceptually)
# Instead: a tight example with active constraint only at one point
# min  x^4 - x^2  s.t. x^2 >= 1  (non-convex — strong duality may not hold)
# We solve both and compare

result_nc_prim = scipy_minimize(lambda x: float(x[0]**4 - x[0]**2),
                       x0=np.array([-1.5]),
                       constraints=[{"type": "ineq", "fun": lambda x: float(x[0]**2 - 1.0)}],
                       method="SLSQP")
p_nc = float(result_nc_prim.fun)

# Dual: max_lambda g(lambda) = max_lambda inf_x [x^4 - x^2 - lambda*(x^2-1)]
# This is a concave function of lambda, maximise numerically
def nc_dual_fn(lam_val: float) -> float:
    '''Dual function for non-convex example: g(lambda) = inf_x [x^4 - x^2 - lam*(x^2-1)].'''
    res_inner = scipy_minimize(
        lambda x: float(x[0]**4 - x[0]**2 - lam_val * (x[0]**2 - 1.0)),
        x0=np.array([0.0]), method="SLSQP",
    )
    return float(res_inner.fun)

lam_range_nc = np.linspace(0.0, 3.0, 60)
dual_vals_nc = np.array([nc_dual_fn(float(lv)) for lv in lam_range_nc])
d_nc = float(dual_vals_nc.max())
gap_nc = abs(p_nc - d_nc)

print("Non-convex example: min x^4 - x^2  s.t. x^2 >= 1")
print(f"  Primal optimum p*    : {p_nc:.5f}")
print(f"  Dual optimum d*      : {d_nc:.5f}")
print(f"  Duality gap |p*-d*|  : {gap_nc:.5f}")
print("")
print("Duality gap is non-zero for the non-convex instance — strong duality fails.")
print("  -> Never apply dual methods blindly to non-convex problems.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Convexity is the central assumption.** Convex sets (closed under convex combinations)
   and convex functions (chord lies above the curve) enable global guarantees that
   are impossible for general non-linear problems.

2. **Lagrangian duality converts constrained problems to unconstrained.** The dual function
   is always concave; the dual problem is always convex — even when the primal is not.

3. **KKT conditions are the language of optimal solutions.** Stationarity,
   primal/dual feasibility, and complementary slackness are necessary for any problem
   and sufficient (with Slater's condition) for convex problems.

4. **SVM duality enables the kernel trick.** The dual objective depends only on inner
   products $\mathbf{x}_i^\top \mathbf{x}_j$, which can be replaced by any
   kernel $K(\mathbf{x}_i, \mathbf{x}_j)$ for non-linear boundaries — without
   explicitly mapping to high-dimensional feature spaces.

5. **Strong duality requires convexity + Slater's condition.** Without these,
   the duality gap may be positive — always verify before applying dual methods.

### What's Next

→ **04-09 (Calibration & Uncertainty Quantification)** applies reliability diagrams,
ECE, temperature scaling, and conformal prediction to ensure that predicted probabilities
reflect true likelihoods.

### Going Further

- Boyd & Vandenberghe (2004). *Convex Optimization.* Cambridge University Press.
  (Chapter 5 for Lagrangian duality, Chapter 11 for interior-point methods.)
- Smola & Schölkopf (2004). *A tutorial on support vector regression.* Statistics and Computing.
- CS229 Lecture Notes (Andrew Ng) — Lecture 6: Support Vector Machines.


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 72)
print("04-08 CONVEX OPTIMIZATION FOUNDATIONS — SUMMARY")
print("=" * 72)
print("")
print("1. Convexity checks (Jensen + second-order)")
print("   x^2, exp(x), |x|, max(0,x): ALL convex — Jensen satisfied")
print("   -x^2, sin(x): NOT convex — Jensen violated for some pairs")
print("")
print("2. Lagrangian duality: min x1^2+x2^2  s.t. x1+x2=1")
print(f"   Primal p*: {f_star_qp:.6f}  |  Dual d*: {-0.5*nu_star_qp**2 - nu_star_qp:.6f}  |  Gap: {abs(f_star_qp - (-0.5*nu_star_qp**2 - nu_star_qp)):.2e}")
print("")
print("3. LP primal and dual (min -x1-2x2 s.t. x1+x2<=4, x1<=2)")
print(f"   Primal p* = {p_star_lp:.5f}   (expected -6.0)")
print("")
print("4. SVM dual (soft margin, C=1.0, linear kernel)")
print(f"   Total SVs   : {n_sv_total} / {len(y_tr_svm)}")
print(f"   Dual obj W* : {dual_obj:.5f}")
print(f"   Primal obj  : {primal_obj:.5f}")
print(f"   Duality gap : {abs(primal_obj - dual_obj):.2e}")
print(f"   Test accuracy (dual vs sklearn): {acc_dual:.4f} vs {acc_sklearn:.4f}")
print(f"   Stationarity residual ||w - sum(ai*yi*xi)||: {stat_residual:.2e}")
print("")
print("5. Slater condition and duality gap")
print(f"   LP (convex, Slater holds): gap = 0 (strong duality)")
print(f"   Non-convex example      : gap = {gap_nc:.5f} (strong duality fails)")
print("")
print("=" * 72)
print("Central insight: for convex problems + Slater, p* = d* = KKT solution.")
print("SVM dual reveals the kernel trick: only inner products appear in W(alpha).")
print("=" * 72)
assert abs(f_star_qp - 0.5) < 1e-4, "QP optimum should be 0.5"
assert p_star_lp < -5.99, "LP optimum should be near -6"
assert acc_dual >= 0.80, "SVM dual accuracy should be >= 80%"
print("All sanity assertions passed.")
